# Full-row temporal feature selection

This example uses synthetic development data with signal, redundancy, interactions, a small business segment, and drift. It demonstrates optional warm-up, memetic NSGA-II, RFE, local refinement, reports, selection, full-development refitting, and checkpoint recovery. It never accesses independent OOT data. Install with `python -m pip install -e ".[dev]"` before running.

In [1]:
from pathlib import Path
import tempfile
import pandas as pd
from IPython.display import display
from memetic_fs import MemeticNSGA2Selector
from memetic_fs.synthetic import make_synthetic_data

root = Path.cwd()
if root.name == "examples":
    root = root.parent
(root / "runs").mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="notebook_", dir=root / "runs"))
frame = make_synthetic_data(n_rows=150000, 
                            n_features=100,
                            n_months=10, 
                            random_state=42, 
                            n_signal_features=28,
                            n_redundant_features=4,
                            n_interaction_features=4,
                            n_segmentation_specific_features=6)
display(frame.head())

,signal_0,signal_1,signal_2,signal_3,signal_4,signal_5,signal_6,signal_7,signal_8,signal_9,...,noise_53,noise_54,noise_55,noise_56,noise_57,target,reference_month,applicant_id,customer_type,age_group
0,0.304717,-1.039984,0.750451,0.940565,-1.951035,-1.302179,0.127840,-0.316243,-0.016801,-0.853044,...,-1.447112,-1.322700,-0.997247,0.399774,-0.905479,0,202301,0,client,60+
1,-0.378163,1.299228,-0.356264,0.737516,-0.933618,-0.205438,-0.950022,-0.339033,0.840308,-1.727320,...,-0.894727,0.643327,-0.394605,-0.005122,-0.163443,1,202301,1,client,46-60
2,0.337575,1.407482,0.090585,0.643939,-2.050172,-0.048718,-0.843230,-1.218813,-0.878152,-0.334123,...,0.176512,0.295994,-0.371915,-1.756722,0.327995,1,202301,2,client,46-60
3,1.727350,-1.533861,0.863828,-0.328525,-0.061324,-1.052899,-0.334456,1.300045,0.582655,1.732312,...,0.138241,0.760133,0.229211,0.530065,-0.704673,1,202301,3,client,46-60
4,-0.179611,0.196776,0.820528,-0.393741,0.521167,-0.265839,-0.117542,0.829519,-1.993060,-1.296472,...,0.935547,-1.831406,-0.335607,-1.990812,-1.495061,0,202301,4,client,31-45


Every observed month is complete. Default CV trains on the first three months and expands; each validation period follows training. Metadata is excluded from predictors, IDs must be unique, and the small prospect segment remains descriptive when its event support is insufficient for a configured hard constraint.

## Optional warm-up and suggested settings

The warm-up validates this development data, inspects the temporal folds and current resources, and pilots fixed LightGBM settings on the three most recent folds. It writes a separate YAML file and **does not start feature selection**. On this 150,000-row example, it can take substantial time; the warning printed before the pilot shows the planned fit count. Set `depth="quick"` for diagnostics without model fits, or `depth="deep"` for all folds and a fuller optimization recommendation. Pilot scores are exploratory and cannot establish the correct number of signal features.

In [2]:
warmup = MemeticNSGA2Selector.warmup_settings_params(
    frame,
    "target",
    depth="deep",
    output_path=run_dir / "warmup_selector.yaml",
    id_col="applicant_id",
    segment_cols=["customer_type", "age_group"],
)

C:\Users\tms19\AppData\Local\Temp\ipykernel_105372\2107502498.py:1: UserWarning: Warm-up depth=deep: 182 planned LightGBM fits on 150000 rows and 100 predictors. Duration depends on data size and the selected depth; there is no time or fit-count cutoff.
  warmup = MemeticNSGA2Selector.warmup_settings_params(


MemoryError: Insufficient memory headroom for warm-up pilot or later full-feature fit

In [ ]:
print("Suggested settings:", warmup.settings_path)
print("Planned pilot fits:", warmup.diagnostics["planned_fits"])
print("Chosen LightGBM settings:", warmup.pilots["selected_model"])

display(pd.DataFrame([
    {"mean_gini": candidate["mean_gini"],
     "elapsed_seconds": candidate["elapsed_seconds"],
     "num_boost_round": candidate["num_boost_round"],
     **candidate["params"]}
    for candidate in warmup.pilots["model_candidates"]
]).sort_values("mean_gini", ascending=False))

display(pd.DataFrame(warmup.pilots["subset_scores"])[["n_features", "mean_gini"]])

display(pd.DataFrame([
    {"setting": setting, "reason": change["reason"]}
    for setting, change in warmup.changes.items()
]))

print("Review the suggested YAML before starting the search.")

The generated YAML is reviewable before the search. `from_config()` loads its proposed values; an inline argument takes precedence over the YAML, and keys inside `optimization`, `resources`, and `model_params` merge with the file. Here we cap the search at 30 generations while using the warm-up's model, feature-range, worker, thread, RFE, and selection suggestions. The tighter Gini tolerance helps avoid accepting a substantially weaker smallest subset, but the final Pareto frontier still needs review.

At `verbose=1`, the notebook shows milestones and a generation progress bar with current worker counts. Set `verbose=2` for routine batch and candidate/fold detail or `verbose=0` for warnings and errors only. Full parent and worker logs are always saved under the experiment `logs/` directory.

In [ ]:
selector = MemeticNSGA2Selector.from_config(
    warmup.settings_path,
    checkpoint_dir=run_dir,
    optimization={"max_generations": 30},
)

In [ ]:
selector.fit(
    frame,
    "target",
    id_col="applicant_id",
    segment_cols=["customer_type", "age_group"],
)

In [ ]:
display(selector.pareto_front_[["key", "n_features", "mean_gini", "std_gini", "min_gini"]])
print("Selected:", selector.best_features_)
print("Stopped:", selector.stop_reason_)

Final comparisons and convergence use full canonical CV. Reduced-fold results have a separate provisional archive. The objective is equal-fold mean Gini (`2*AUC-1`); pooled out-of-fold Gini and temporal stability are diagnostics.

In [ ]:
display(selector.temporal_performance_)
display(selector.segment_performance_.head(10))
display(selector.selection_frequency_.head(12))
display(selector.history_)
assert {r["fidelity"] for r in selector.confirmed_} == {"full"}
assert all(len(r["folds"]) == len(selector.folds_) for r in selector.confirmed_)
assert selector.history_["local_search_evaluations"].sum() > 0
assert selector.rfe_trajectory_

The offline report includes the Pareto approximation, cardinality/RFE trajectory, feature frequency, co-selection, temporal Gini, segment metrics, and evolution history. CSV/Parquet/JSON exports accompany it.

In [ ]:
from memetic_fs.persistence.checkpoints import read_json
from memetic_fs.reporting.visualizations import build_figures
payload = read_json(run_dir / "results.json")
figures = build_figures(payload)
print("Charts:", list(figures))
print("Offline report:", run_dir / "report.html")
print("Exports:", sorted(p.name for p in run_dir.glob("*.parquet")))

Selection policies include tolerance, geometric knee, manual candidate choice, and an approximate paired temporal-block uncertainty rule. With few validation folds, statistical evidence is weak. Here the chosen manual candidate is then refitted on all development rows. External generalization assessment remains separate.

In [ ]:
selected_id = selector.selected_candidate_["key"]
assert selector.select("manual", candidate_id=selected_id)
selected_data = selector.transform(frame)
model = selector.fit_final_model()
assert model.num_feature() == selected_data.shape[1]
print("Final model features:", selected_data.shape[1])
print("Final model:", run_dir / "final_model.txt")

Resume verifies persisted input digests and evaluator settings. Completed model evaluations are reused. JSON optimizer checkpoints avoid executable pickle deserialization.

In [ ]:
completed_before = len(selector.evaluator_.cache.completed())
restored = MemeticNSGA2Selector.resume(run_dir)
assert len(restored.evaluator_.cache.completed()) == completed_before
assert [r["mask"] for r in restored.confirmed_] == [r["mask"] for r in selector.confirmed_]
print("Reused completed evaluations:", completed_before)
print("Confirmed frontier size:", len(restored.confirmed_))

Repeated subset search creates model-selection optimism. This frontier describes evaluated candidates and has no global-optimality guarantee. Run independent OOT assessment outside this framework. For production scale, use the separate `benchmarks/full_row.py` utility; this notebook does not establish feasibility for 1.5 million rows.